In [1]:
!pip install -q "transformers>=4.40.0" datasets peft accelerate bitsandbytes tqdm



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 28.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 35.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━

In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

2025-11-24 18:24:44.190365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764008684.390054      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764008684.447074      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
import os
if "COLAB_GPU" in os.environ:
  from google.colab import output
  output.enable_custom_widget_manager()

In [ ]:
import os
from huggingface_hub import notebook_login

# If running in Google Colab
if "COLAB_GPU" in os.environ:
    !huggingface-cli login
# If running locally (Jupyter, VS Code, etc.)
else:
    notebook_login() 


In [5]:
dataset = load_dataset("Amod/mental_health_counseling_conversations")


README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [6]:
dataset = dataset["train"].train_test_split(test_size=0.30, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]


In [7]:


# 3. Convert to pandas
train_df = train_dataset.to_pandas()
test_df  = test_dataset.to_pandas()

# 4. Reset index
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)



In [8]:
 def format_prompt(example):
     return {
         "text": f"<s>Context: {example['Context']}\nResponse: {example['Response']}</s>"
#f"<s>Context: {example['Context']}\nResponse: {example['Response']}"
     }

# def format_example(example):
#     text = f"<s>Context: {example['Context']}\nResponse: {example['Response']}</s>"
#     return {"text": text}

train_dataset = train_dataset.map(format_prompt)
test_dataset  = test_dataset.map(format_prompt)

Map:   0%|          | 0/2458 [00:00<?, ? examples/s]

Map:   0%|          | 0/1054 [00:00<?, ? examples/s]

In [ ]:

# TOKENIZER

model_name = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# LLaMA tokenizer normally has no pad_token so  eos_token instead
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

max_length = 512  # adjust if needed


def tokenize(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
    enc["labels"] = enc["input_ids"].copy()  # causal LM labels
    return enc


train_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
test_tokenized  = test_dataset.map(tokenize,  batched=True, remove_columns=test_dataset.column_names)


train_tokenized.set_format("torch")
test_tokenized.set_format("torch")

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Map:   0%|          | 0/2458 [00:00<?, ? examples/s]

Map:   0%|          | 0/1054 [00:00<?, ? examples/s]

In [10]:
print(train_tokenized[0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:

#  QLoRA 4-BIT QUANTIZATION

# Loads model in 4-bit NF4 format
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

In [ ]:
# LOAD MODEL IN 4-BIT

from transformers import AutoModelForCausalLM

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)


model.config.use_cache = False

model = prepare_model_for_kbit_training(model)




config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [ ]:
# LoRA CONFIGURATION

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


In [ ]:
output_dir   = "./qlora-llama32-mental-health"
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=50,

    eval_steps=300,        # evaluate every 300 steps
    eval_strategy="epoch",
    save_strategy="epoch", 

    # save_steps=300,        # save every 300 steps
    # save_total_limit=2,

    bf16=True,
    report_to="none",
)


In [ ]:

training_args.label_names = ["labels"]

In [ ]:
# DATA COLLATOR + TRAINER

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
)
# TRAIN

trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.274100,2.234400
2,1.939600,2.031966
3,1.679800,1.970443


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=462, training_loss=2.040167036510649, metrics={'train_runtime': 8205.3886, 'train_samples_per_second': 0.899, 'train_steps_per_second': 0.056, 'total_flos': 2.229995696239411e+16, 'train_loss': 2.040167036510649, 'epoch': 3.0})

In [17]:
# Save LoRA adapter + tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ QLoRA fine-tuning finished. Adapter saved to:", output_dir)

✅ QLoRA fine-tuning finished. Adapter saved to: ./qlora-llama32-mental-health


In [23]:
from huggingface_hub import login
login()

In [ ]:
import shutil

folder_path = "./qlora-llama32-mental-health"
zip_path = "./qlora-llama32-mental-health.zip"

shutil.make_archive(zip_path.replace(".zip",""), 'zip', folder_path)

print("✔ Folder zipped successfully:", zip_path)


In [ ]:
# #upload login token
# import os
# from huggingface_hub import notebook_login

# # If running in Google Colab
# if "COLAB_GPU" in os.environ:
#     !huggingface-cli login
# # If running locally (Jupyter, VS Code, etc.)
# else:
#     notebook_login() 


In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="./qlora-llama32-mental-health", 
    repo_id="ZunairAhmad/llama3.2_mental_health",
    repo_type="model"
)

print("✔ Uploaded successfully!")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✔ Uploaded successfully!


In [27]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("ZunairAhmad/llama3.2_mental_health", repo_type="model")

print("Files in repo:")
for f in files:
    print(" -", f)

Files in repo:
 - .gitattributes
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - checkpoint-154/README.md
 - checkpoint-154/adapter_config.json
 - checkpoint-154/adapter_model.safetensors
 - checkpoint-154/optimizer.pt
 - checkpoint-154/rng_state.pth
 - checkpoint-154/scheduler.pt
 - checkpoint-154/special_tokens_map.json
 - checkpoint-154/tokenizer.json
 - checkpoint-154/tokenizer_config.json
 - checkpoint-154/trainer_state.json
 - checkpoint-154/training_args.bin
 - checkpoint-308/README.md
 - checkpoint-308/adapter_config.json
 - checkpoint-308/adapter_model.safetensors
 - checkpoint-308/optimizer.pt
 - checkpoint-308/rng_state.pth
 - checkpoint-308/scheduler.pt
 - checkpoint-308/special_tokens_map.json
 - checkpoint-308/tokenizer.json
 - checkpoint-308/tokenizer_config.json
 - checkpoint-308/trainer_state.json
 - checkpoint-308/training_args.bin
 - checkpoint-462/README.md
 - checkpoint-462/adapter_config.json
 - checkpoint-462/adapter_model.safetensors
 - check